# TSM long Development Snapshot v1.2 — preparation only

[Open in Colab](https://colab.research.google.com/github/hh4832/market-research-core/blob/feature/phase4a-tsm-native-research-adapter/notebooks/TSM_Long_Development_Snapshot_Preparation.ipynb)

Run top to bottom in a fresh runtime. Only the research lead may execute the
personal approval cell. Planning reads metadata only. Full source hashing reads
Holdout bytes; mixed row-group decoding may materialize Holdout records before
filtering. No approval means an exception before either operation or output creation.
Never paste a previous one-month plan approval. No Agent, strategy, metrics,
research authorization or budget operation is invoked. Do not display source OHLCV.


In [ ]:
from pathlib import Path
CORE_SHA = "7bcc61813a6cf2956c6df69ea796bf1c7f2b495f"
SOURCE_ROOT = Path('/content/drive/MyDrive/00Quant_Research/data/00_Quant_Research_Hub/20261101_205431_06359faf7ef4_IBKR_FINAL_GATE_FULL')
PARENT_MANIFEST_SHA = "d960602cff24f7350b056d69e19d8e0b8734cdad266d3782bc56ebc507818828"
OUTPUT_PARENT = Path('/content/drive/MyDrive/00Quant_Research/data/development_snapshots')
REPORT_PARENT = Path('/content/drive/MyDrive/00Quant_Research/phase4a_preparation_reports')


In [ ]:
import subprocess, sys, json, re, importlib
from datetime import datetime, timezone
assert re.fullmatch('[0-9a-f]{40}', CORE_SHA)
CORE_ROOT = Path('/content/market-research-core-' + CORE_SHA)
if not CORE_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/hh4832/market-research-core.git', str(CORE_ROOT)], check=True)
    subprocess.run(['git', '-C', str(CORE_ROOT), 'checkout', '--detach', CORE_SHA], check=True)
assert subprocess.check_output(['git', '-C', str(CORE_ROOT), 'rev-parse', 'HEAD'], text=True).strip() == CORE_SHA
assert not subprocess.check_output(['git', '-C', str(CORE_ROOT), 'status', '--porcelain'], text=True).strip()
# Existing declared dependencies only, in the user's fresh Colab runtime.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(CORE_ROOT)], check=True)
sys.path.insert(0, str(CORE_ROOT / 'src'))
importlib.invalidate_caches()
from market_research_core.tsm_native import core_code_identity
from market_research_core.tsm_development import (
    LONG_WINDOW, DevelopmentSnapshotPin, plan_conversion, produce_development_snapshot,
    preflight_development_snapshot, renew_development_receipt, file_sha, digest,
)
identity = core_code_identity()
assert identity['core_git_sha'] == CORE_SHA and identity['core_tree_dirty'] is False, identity
from google.colab import drive
drive.mount('/content/drive')
REPORT_ROOT = REPORT_PARENT / ('long_v12_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
REPORT_ROOT.mkdir(parents=True, exist_ok=False)
def save(name, value):
    with (REPORT_ROOT / name).open('x') as f:
        json.dump(value, f, indent=2, allow_nan=False)
        f.write('\n')
save('environment.json', {'core_sha': CORE_SHA, 'strategy_run': False, 'batch_consumption': 0})


In [ ]:
# Footer-only source planning. The published source SHA is a claim until approved hashing.
manifest = SOURCE_ROOT / 'final_gate_manifest.json'
assert file_sha(manifest) == PARENT_MANIFEST_SHA, 'STOP: parent manifest mismatch'
parent = json.loads(manifest.read_text())
tsm = [r for r in parent['results'] if r.get('symbol') == 'TSM']
assert len(tsm) == 1, 'STOP: ambiguous source lineage'
source_claim = tsm[0]['published_sha256']
assert re.fullmatch('[0-9a-f]{64}', source_claim)
plan = plan_conversion([{'path': str(SOURCE_ROOT / 'tsm_normalized.parquet'),
    'expected_sha256': source_claim,
    'lineage_ref': str(manifest) + '#TSM.published_sha256 (claim until independently hashed)'}],
    parent_manifest_path=manifest, expected_parent_manifest_sha256=PARENT_MANIFEST_SHA,
    window=LONG_WINDOW)
assert digest({k: v for k, v in plan.items() if k != 'plan_sha256'}) == plan['plan_sha256']
save('producer_plan.json', plan)
print(json.dumps(plan, indent=2))
print('Plan SHA256:', plan['plan_sha256'])
print('Holdout bytes in source hashing:', any(s['opaque_hash_includes_holdout_bytes'] for s in plan['sources']))
print('Possible Holdout record decoding:', any(g['decode_may_include_holdout'] for s in plan['sources'] for g in s['selected_row_groups']))
print('Output Holdout rows:', plan['output_holdout_rows'])
print('Source SHA claim (not yet independently verified):', source_claim)


In [ ]:
# Research lead PERSONAL ACTION ONLY. No default approval and no assistant execution.
# Require explicit evidence even if future metadata reports no mixed source risk.
approval = None
approval_ref = input('Research lead: enter HTTPS approval evidence reference (blank stops): ').strip()
confirmation = input('Research lead: personally type APPROVE ' + plan['plan_sha256'] + ' (blank stops): ').strip()
if not approval_ref.startswith('https://') or confirmation != 'APPROVE ' + plan['plan_sha256']:
    raise RuntimeError('STOP: no personal exact-plan approval; no source hash/decode/output performed')
approval = {'status': 'APPROVED', 'approved_by': 'hh4832', 'approval_ref': approval_ref,
    'plan_sha256': plan['plan_sha256'], 'allow_full_source_opaque_hash': True,
    'allow_holdout_decode_for_conversion_only': True, 'allow_strategy_run': False,
    'approved_at': datetime.now(timezone.utc).isoformat(),
    'confirmation_method': 'PERSONALLY_TYPED_EXACT_PLAN_SHA_IN_COLAB'}
save('producer_approval.PERSONAL.json', approval)


In [ ]:
# This cell also fails closed if the prior cell was skipped or declined.
assert approval is not None and approval['plan_sha256'] == plan['plan_sha256'], 'STOP: approval required'
bundle = produce_development_snapshot(plan, OUTPUT_PARENT, producer_sha=CORE_SHA, approval=approval)
save('produced_binding.json', bundle)
root = Path(bundle['snapshot_root'])
manifest_doc = json.loads((root / 'manifest.json').read_text())
lineage = json.loads((root / 'lineage.json').read_text())
pin = DevelopmentSnapshotPin(**bundle['snapshot_pin'])
assert file_sha(root / 'manifest.json') == bundle['manifest_sha256']
assert file_sha(root / 'lineage.json') == pin.lineage_sha256 == manifest_doc['lineage_sha256']
assert manifest_doc['snapshot_id'] == pin.snapshot_id
assert manifest_doc['window_identity'] == lineage['window_identity'] == plan['window_identity']
assert manifest_doc['output_holdout_rows'] == lineage['output_holdout_rows'] == 0
report = preflight_development_snapshot(bundle)
assert report['status'] == 'PASS' and report['output_holdout_rows'] == 0
# Renewal reads ONLY the new development file, never the mixed source.
renewed = renew_development_receipt(bundle)
assert preflight_development_snapshot(renewed)['status'] == 'PASS'
save('renewed_binding.json', renewed)
save('verified_preflight.json', report)
save('summary.json', {'snapshot_id': pin.snapshot_id, 'output_holdout_rows': 0,
    'core_sha': CORE_SHA, 'plan_sha256': plan['plan_sha256'],
    'conversion_holdout_decode_occurred': any(s['holdout_decode_occurred'] for s in lineage['sources']),
    'strategy_run': False, 'batch_consumption': 0, 'research_authorization': 'NOT_CREATED'})
print('Development-only snapshot archived:', root)
print('Exclusive reports archived:', REPORT_ROOT)
print('STOP. No research authorization or strategy run is performed.')


## Stop

A successful output proves Development-only records; it does not erase the
approved source hashing or mixed-group decoding exposure recorded in lineage.
All artifacts are already archived in new Google Drive paths. Never overwrite an
existing snapshot. Real execution is external evidence; synthetic CI is not a
real-data approval or preflight result. Do not proceed to strategy research here.
